# 03 — EDA and Balance Check
Vellakkal Dumpsite methane hotspot project.

Runs `src/preprocessing.py` to build the stacked `patches_array.npz` dataset, then explores class balance, spatial distribution, and a visual sample of patches before moving to splitting and training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/methane-hotspot-detection'

In [ ]:
!pip install -q rasterio tqdm

## Build the stacked dataset
Requires the exported GeoTIFFs from Batch 1 to already be downloaded/synced into `data/raw/patches_geotiff/` (Drive export folder), and `metadata.csv` from Batch 2.

In [ ]:
!python {PROJECT_ROOT}/src/preprocessing.py \
    --geotiff-dir {PROJECT_ROOT}/data/raw/patches_geotiff \
    --metadata {PROJECT_ROOT}/data/processed/metadata.csv \
    --out {PROJECT_ROOT}/data/processed/patches_array.npz \
    --patch-size 32

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = np.load(f'{PROJECT_ROOT}/data/processed/patches_array.npz', allow_pickle=True)
X, y, patch_ids = data['X'], data['y'], data['patch_ids']
meta = pd.read_csv(f'{PROJECT_ROOT}/data/processed/metadata_final.csv')

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Genuine, non-duplicated patch count:', len(patch_ids))

## Class balance

In [ ]:
counts = pd.Series(y).value_counts().sort_index()
print(counts)
counts.plot(kind='bar', title='Hotspot (1) vs Non-hotspot (0) — genuine patch counts')
plt.xlabel('label_enhancement')
plt.ylabel('count')
plt.show()

## Spatial distribution of samples and labels
Confirms coverage around the dumpsite and shows whether hotspot-labeled patches cluster spatially (expected, given the coarse CH4 footprint) — informs how Batch 4 should draw block boundaries.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(meta['longitude'], meta['latitude'], c=meta['label_enhancement'],
                 cmap='coolwarm', s=15, alpha=0.7)
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title('Spatial distribution of labeled patches')
plt.colorbar(sc, label='label_enhancement')
plt.show()

## CH4 enhancement distribution
Sanity check that the enhancement-based threshold is producing a sensible separation, not just noise.

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(meta.loc[meta['label_enhancement']==0, 'ch4_enhancement'], bins=30, alpha=0.6, label='non-hotspot')
plt.hist(meta.loc[meta['label_enhancement']==1, 'ch4_enhancement'], bins=30, alpha=0.6, label='hotspot')
plt.xlabel('CH4 enhancement over local background (ppb)')
plt.ylabel('count')
plt.legend()
plt.title('Enhancement distribution by label')
plt.show()

## Visual sample of genuine patches (true color)
A quick human sanity check — do hotspot-labeled patches look like plausible dumpsite/bare-ground/waste areas, and non-hotspot patches like vegetation/background land cover?

In [ ]:
def show_patch(idx, ax):
    # Channels: B2,B3,B4,B8,B11,B12,NDVI -> true color uses B4,B3,B2 (indices 2,1,0)
    rgb = X[idx][..., [2, 1, 0]]
    rgb = np.clip(rgb / rgb.max(), 0, 1) if rgb.max() > 0 else rgb
    ax.imshow(rgb)
    ax.set_title(f"{'hotspot' if y[idx]==1 else 'non-hotspot'}\n{patch_ids[idx]}")
    ax.axis('off')

hotspot_idx = np.where(y == 1)[0][:4]
nonhotspot_idx = np.where(y == 0)[0][:4]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, idx in enumerate(hotspot_idx):
    show_patch(idx, axes[0, i])
for i, idx in enumerate(nonhotspot_idx):
    show_patch(idx, axes[1, i])
plt.tight_layout()
plt.show()

## Report the honest final numbers
This is the number to report in the project write-up — no padding, no duplicated rows.

In [ ]:
print(f'Genuine, non-duplicated 2D CNN patches: {len(patch_ids)}')
print(f'Unique locations represented: {meta[["latitude","longitude"]].drop_duplicates().shape[0]}')
print(f'Unique acquisition dates represented: {meta["patch_date"].nunique()}')
print(f'Unique CH4 source pixels represented: {meta["ch4_pixel_id"].nunique()}')
print(f'Class balance: {pd.Series(y).value_counts(normalize=True).round(3).to_dict()}')